In [ ]:
import os
import operator
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
model = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
os.environ['NO_PROXY'] = '*'

llm = ChatOpenAI(
    model=model,
    api_key=api_key,
)
 
MAX_ROUNDS = 3

In [ ]:
#FILL

left_system = SystemMessage(content=
    "System prompt for a left leaning candidate"
)

In [ ]:
#FILL 

right_system = SystemMessage(content=
    "System prompt for a right leaning candidate"
)

In [ ]:
judge_system = SystemMessage(content=
    "You are a neutral debate judge with no political bias. "
    "Evaluate purely on logic, evidence, and argumentation quality."
    "Keep your evaluation to at most 200 words"
    "Finally decide who has the stronger argument"
)

In [ ]:
# Agent like prompt generation 

def left_agent(prompt: str) -> str:
    return llm.invoke([left_system, HumanMessage(content=prompt)]).content
 
def right_agent(prompt: str) -> str:
    return llm.invoke([right_system, HumanMessage(content=prompt)]).content
 
def judge_agent(prompt: str) -> str:
    return llm.invoke([judge_system, HumanMessage(content=prompt)]).content

In [ ]:
# States

class State(TypedDict):
    topic:     str
    arguments: Annotated[list, operator.add]
    round:     int
    verdict:   str

In [ ]:
# First node

def opening_statements(state: State) -> State:
    topic = state["topic"]
    print("\n--- Opening Statements ---")
 
    left = left_agent(f"Give an opening statement on: {topic}. 3 sentences.")
    print(f"Left:  {left}\n")
 
    right = right_agent(f"Give an opening statement on: {topic}. 3 sentences.")
    print(f"Right: {right}\n")
 
    return {"arguments": [f"[Opening - Left]: {left}", f"[Opening - Right]: {right}"]}

In [ ]:
# Rounds in between

def argue(state: State) -> State:
    round_num = state["round"] + 1
    topic     = state["topic"]
    history   = "\n".join(state["arguments"])
    print(f"--- Round {round_num} ---")
 
    left = left_agent(
        f"Topic: {topic}\n\n"
        f"Debate so far:\n{history}\n\n"
        f"Your round {round_num} argument. 3 sentences."
    )
    print(f"Left:  {left}\n")
 
    right = right_agent(
        f"Topic: {topic}\n\n"
        f"Debate so far:\n{history}\n\n"
        f"Left just said: {left}\n\n"
        f"Your round {round_num} counter. 3 sentences."
    )
    print(f"Right: {right}\n")
 
    return {
        "arguments": [f"[Round {round_num} - Left]: {left}", f"[Round {round_num} - Right]: {right}"],
        "round": round_num,
    }

In [ ]:
def conclude(state: State) -> State:
    topic   = state["topic"]
    history = "\n".join(state["arguments"])
    print("--- Closing Statements ---")
 
    left = left_agent(f"Topic: {topic}\n\nGive a closing statement. 3 sentences.")
    print(f"Left:  {left}\n")
 
    right = right_agent(f"Topic: {topic}\n\nGive a closing statement. 3 sentences.")
    print(f"Right: {right}\n")
 
    print("--- Verdict ---")
    verdict = judge_agent(
        f"Topic: {topic}\n\n"
        f"Full debate:\n{history}\n\n"
        f"Closing - Left:  {left}\n"
        f"Closing - Right: {right}\n\n"
        "Who made the stronger case and why? Be specific."
    )
    print(f"Verdict: {verdict}\n")
    return {"verdict": verdict}

In [ ]:

def should_continue(state: State) -> str:
    return "argue" if state["round"] < MAX_ROUNDS else "conclude"

In [ ]:
graph = StateGraph(State)
graph.add_node("opening",  opening_statements)
graph.add_node("argue",    argue)
graph.add_node("conclude", conclude)
 
graph.set_entry_point("opening")
graph.add_edge("opening", "argue")
graph.add_conditional_edges("argue", should_continue)
graph.add_edge("conclude", END)
 
app = graph.compile()

In [ ]:
topic = input("Enter a debate topic: ")
print(f"\n{'='*60}\nDEBATE: {topic}\n{'='*60}")
 
app.invoke({
    "topic":     topic,
    "arguments": [],
    "round":     0,
    "verdict":   "",
})